In [10]:
!pip install modal

In [11]:
import os
from google.colab import userdata

try:
    os.environ["MODAL_TOKEN_ID"] = userdata.get('token-id')
    os.environ["MODAL_TOKEN_SECRET"] = userdata.get('token-secret')
    print("✅ Authentifizierung für Modal geladen!")
except Exception as e:
    print(f"❌ Fehler: {e}. Hast du das Schlüssel-Icon links konfiguriert?")

tid = os.environ.get("MODAL_TOKEN_ID")
tsec = os.environ.get("MODAL_TOKEN_SECRET")
print(f"Token ID startet mit: {tid[:4]}... (Länge: {len(tid)})")
print(f"Token Secret startet mit: {tsec[:4]}... (Länge: {len(tsec)})")



✅ Authentifizierung für Modal geladen!
Token ID startet mit: ak-b... (Länge: 25)
Token Secret startet mit: as-p... (Länge: 25)


In [12]:
import os
from google.colab import userdata

# 1. Token aus den Colab-Secrets laden
hf_token_value = userdata.get('HF_TOKEN')

# 2. Das Modal-Secret erstellen, ohne den Token im Code zu zeigen
# Wir nutzen die f-String Syntax für den System-Befehl
if hf_token_value:
    !modal secret create vbot_modal_huggingface HF_TOKEN='{hf_token_value}' --force
    print("✅ Modal Secret 'vbot_modal_huggingface' wurde erfolgreich erstellt!")
else:
    print("❌ Fehler: HF_TOKEN wurde nicht in den Colab-Secrets gefunden.")

Created a new secret 'vbot_modal_huggingface' with the key 'HF_TOKEN'

Use it in your Modal app:

                                                                                
@app.function(secrets=[modal.Secret.from_name("vbot_modal_huggingface")])       
def some_function():                                                            
    os.getenv("HF_TOKEN")                                                       
                                                                                
✅ Modal Secret 'vbot_modal_huggingface' wurde erfolgreich erstellt!


In [13]:

!modal profile list

┏━━━┳━━━━━━━━━┳━━━━━━━━━━━┓
┃   ┃ Profile ┃ Workspace ┃
┡━━━╇━━━━━━━━━╇━━━━━━━━━━━┩
└───┴─────────┴───────────┘
Using matthias-nollek workspace based on environment variables


In [31]:
%%writefile app.py
import modal
import os
import uuid
import asyncio
import json
import time # Added for chat completions endpoint
from fastapi import FastAPI, WebSocket, Request
from starlette.responses import HTMLResponse, StreamingResponse # Added StreamingResponse

# 1. Konfiguration
MAX_TOKENS = 8192
MAX_NEW_TOKENS = 2048
MODEL_ID = "mistralai/Mistral-Nemo-Instruct-FP8-2407"
SYSTEM_PROMPT = "You are a helpful assistant..."

# Define the model path on the volume
MODEL_VOLUME_PATH = "/data/mistral-nemo"

# Create (or reference) a Volume named "model-storage"
model_volume = modal.Volume.from_name("model-storage", create_if_missing=True)

# 2. Image Definition
vllm_image = (
    modal.Image.debian_slim(python_version="3.10")
    .pip_install(
        "vllm>=0.6.0",
        "transformers>=4.44.0",
        "tokenizers>=0.19.0",
        "fastapi",
        "starlette",
        "hf_transfer",
        "huggingface_hub"
    )
    .env({
        "HF_HUB_ENABLE_HF_TRANSFER": "1",
        "VLLM_LOGGING_LEVEL": "ERROR",
        "VLLM_USE_V1": "0"
    })
)

app = modal.App("twilio-voice-nemo")

# 3. Der Haupt-Service
@app.cls(
    image=vllm_image,
    gpu="A10G",
    secrets=[modal.Secret.from_name("vbot_modal_huggingface")],
    volumes={"/data": model_volume},
    scaledown_window=60
)
class TwilioChatBot:
    @modal.enter()
    def load_engine(self):
        from vllm import AsyncEngineArgs, AsyncLLMEngine
        from transformers import AutoTokenizer
        from huggingface_hub import snapshot_download

        print(f"Checking for model at: {MODEL_VOLUME_PATH}")
        # Check if model is on the volume, download if not
        if not os.path.exists(MODEL_VOLUME_PATH):
            print("📥 Model will be downloaded to the volume for the first time...")
            snapshot_download(
                MODEL_ID,
                local_dir=MODEL_VOLUME_PATH,
                ignore_patterns=["*.pt", "*.bin"]
            )
            model_volume.commit()
            print(f"✅ Model downloaded to {MODEL_VOLUME_PATH}. Contents:")
            os.system(f"ls -l {MODEL_VOLUME_PATH}")
        else:
            print(f"✅ Model already exists at {MODEL_VOLUME_PATH}. Contents:")
            os.system(f"ls -l {MODEL_VOLUME_PATH}")

        engine_args = AsyncEngineArgs(
            model=MODEL_VOLUME_PATH,
            gpu_memory_utilization=0.90,
            max_model_len=MAX_TOKENS,
            trust_remote_code=True,
            enforce_eager=True
        )
        self.engine = AsyncLLMEngine.from_engine_args(engine_args)
        self.tokenizer = AutoTokenizer.from_pretrained(MODEL_VOLUME_PATH, local_files_only=True)

    @modal.method()
    async def generate_stream(self, prompt_token_ids: list[int], sampling_params=None):
        from vllm import SamplingParams

        if sampling_params is None:
            # Default SamplingParams if not provided
            sampling_params = SamplingParams(
                temperature=0.7,
                max_tokens=256,
                presence_penalty=0.2
            )

        results_generator = self.engine.generate(prompt_token_ids=prompt_token_ids, sampling_params=sampling_params, request_id=f"req-{os.urandom(4).hex()}")

        last_output_len = 0
        async for request_output in results_generator:
            # vLLM gibt immer den bisher gesamten Text zurück,
            # daher extrahieren wir nur den neuen Teil (den "Delta")
            full_text = request_output.outputs[0].text
            delta = full_text[last_output_len:]
            last_output_len = len(full_text)

            if delta:
                yield delta

    @modal.asgi_app()
    def fastapi_app(self):
        web_app = FastAPI()

        @web_app.post("/start_call")
        async def start_call(request: Request):
            # 1. Dynamische URL-Auflösung über das Request-Objekt
            # Dies extrahiert automatisch den Hostname (z.B. user--app-name.modal.run)
            host = request.url.netloc

            # 2. Das XML-Template mit der korrekten wss:// URL
            # Wir nutzen .format() für Robustheit
            response_xml = """<?xml version=\"1.0\" encoding=\"UTF-8\"?>
        <Response>
          <Connect>
            <ConversationRelay
                url=\"wss://{host}/ws\"
                welcomeGreeting=\"Hi! I'm Jane. Just chat with me!!\">
            </ConversationRelay>
          </Connect>
        </Response>""".format(host=host)

            return HTMLResponse(content=response_xml, media_type="application/xml")

        @web_app.post("/v1/chat/completions") # New endpoint for AnythingLLM or similar
        async def chat_completions_endpoint(request: Request):
            from vllm import SamplingParams # Import locally if not already global

            data = await request.json()
            messages = data.get("messages", [])
            stream_mode = data.get("stream", False) # Check for stream flag

            # Format messages for vLLM chat template
            history_for_llm = []
            for msg in messages:
                history_for_llm.append({"role": msg["role"], "content": msg["content"]})

            # Generate token IDs using the chat template
            prompt_token_ids_for_llm = self.tokenizer.apply_chat_template(
                history_for_llm,
                tokenize=True,
                #add_generation_prompt=True, # Recommended for generation tasks -- REMOVED THIS LINE
            )

            # Sampling parameters from request, with fallbacks to class defaults
            sampling_params_dict = data.get("sampling_params", {})
            sampling_params_for_stream = SamplingParams(
                temperature=sampling_params_dict.get("temperature", 0.7),
                max_tokens=sampling_params_dict.get("max_tokens", MAX_NEW_TOKENS),
                presence_penalty=sampling_params_dict.get("presence_penalty", 0.2)
            )

            async def generate_and_stream_response():
                # OpenAI-like streaming format
                # Each chunk should be a dict like { "choices": [ { "delta": { "content": "token" } } ] }
                # and wrapped in "data: " followed by two newlines
                stream_generator = await self.generate_stream.local(prompt_token_ids_for_llm, sampling_params_for_stream)
                async for token_chunk in stream_generator:
                    yield f"data: {json.dumps({'choices': [{'delta': {'content': token_chunk}}]})}\n\n"
                yield "data: [DONE]\n\n" # Signal end of stream

            if stream_mode:
                return StreamingResponse(generate_and_stream_response(), media_type="text/event-stream")
            else:
                full_response_content = ""
                stream_generator = await self.generate_stream.local(prompt_token_ids_for_llm, sampling_params_for_stream)
                async for token in stream_generator:
                    full_response_content += token

                # Calculate tokens (approximation)
                prompt_tokens = len(prompt_token_ids_for_llm)
                completion_tokens = len(self.tokenizer.encode(full_response_content, add_special_tokens=False))
                total_tokens = prompt_tokens + completion_tokens

                # Return in OpenAI-like chat completion format
                return {
                    "id": f"chatcmpl-{uuid.uuid4().hex}",
                    "object": "chat.completion",
                    "created": int(time.time()),
                    "model": MODEL_ID, # Use MODEL_ID defined at the top
                    "choices": [
                        {
                            "index": 0,
                            "message": {
                                "role": "assistant",
                                "content": full_response_content,
                            },
                            "finish_reason": "stop"
                        }
                    ],
                    "usage": {
                        "prompt_tokens": prompt_tokens,
                        "completion_tokens": completion_tokens,
                        "total_tokens": total_tokens
                    }
                }

        @web_app.websocket("/ws")
        async def websocket_endpoint(websocket: WebSocket):
            await websocket.accept()
            queue = asyncio.Queue()
            # Initialer Verlauf mit System Prompt
            history = [{"role": "system", "content": SYSTEM_PROMPT}]

            async def llm_request(message):
                # Prepare the prompt for the LLM using chat template
                prompt_token_ids_for_llm = self.tokenizer.apply_chat_template(
                    history + [dict(role="user", content=message)],
                    tokenize=True, #add_generation_prompt=True, -- REMOVED THIS LINE
                )
               try:
                    assistant_start_id = self.tokenizer.control_tokens['start_assistant_token_id']
                    prompt_token_ids_for_llm.append(assistant_start_id)
               except KeyError:
                    # Fallback, falls der Tokenizer-Key anders heißt
                    # Bei Mistral Nemo ist das oft die ID 3
                    prompt_token_ids_for_llm.append(3)
                full_reply_tokens = [] # To reconstruct the full reply for history

                try:
                    # Call the generate_stream method (locally within the class instance)
                    # Explicitly await self.generate_stream to ensure it's an async generator
                    stream_to_iterate = await self.generate_stream.local(prompt_token_ids_for_llm)
                    async for token in stream_to_iterate:
                        full_reply_tokens.append(token)
                        # Send each token immediately to Twilio
                        await websocket.send_json({"type": "text", "token": token, "last": False})

                except asyncio.CancelledError:
                    # If the task is cancelled (e.g., by an interrupt from Twilio), re-raise
                    raise
                finally:
                    # Reconstruct the full reply for history and send final message
                    reply = "".join(full_reply_tokens)
                    if reply: # Nur speichern, wenn auch wirklich eine Antwort kam
                    history.append(dict(role="user", content=message))
                    history.append(dict(role="assistant", content=reply))
                    await websocket.send_json({"type": "text", "token": "", "last": True})

            async def read_from_socket():
                async for data in websocket.iter_json():
                    await queue.put(data)

            async def process_logic():
                input_buffer = []
                llm_task = None
                while True:
                    data = await queue.get()
                    if data["type"] == "prompt":
                        input_buffer.append(data["voicePrompt"])
                        if data.get("last"):
                            message = " ".join(input_buffer)
                            input_buffer = []
                            if llm_task: llm_task.cancel() # Cancel previous task if new prompt comes
                            llm_task = asyncio.create_task(llm_request(message))
                    elif data["type"] == "interrupt":
                        input_buffer = [] # Clear buffer on interrupt
                        if llm_task: llm_task.cancel() # Cancel current LLM task on interrupt

            await asyncio.gather(read_from_socket(), process_logic())

        return web_app

Overwriting app.py


In [32]:
!python3 -m py_compile app.py

In [33]:
!modal deploy app.py

⠸ Creating objects...
⠦ Creating objects...
⠏ Creating objects...
⠙ Creating objects...
├── 🔨 Created mount /content/app.py
├── 🔨 Created function TwilioChatBot.*.
└── 🔨 Created web endpoint for TwilioChatBot.fastapi_app => 
    https://matthias-nollek--twilio-voice-nemo-twiliochatbot-fastapi-app.modal.r
    un
✓ Created objects.
├── 🔨 Created mount /content/app.py
├── 🔨 Created function TwilioChatBot.*.
└── 🔨 Created web endpoint for TwilioChatBot.fastapi_app => 
    https://matthias-nollek--twilio-voice-nemo-twiliochatbot-fastapi-app.modal.r
    un
✓ App deployed in 1.238s! 🎉

View Deployment: 
https://modal.com/apps/matthias-nollek/main/deployed/twilio-voice-nemo


In [34]:
!curl -X POST https://matthias-nollek--twilio-voice-nemo-twiliochatbot-fastapi-app.modal.run/start_call \
     -H "Content-Type: application/json" \
     -d '{"prompt": "Wer bist du?"}'

<?xml version="1.0" encoding="UTF-8"?>
        <Response>
          <Connect>
            <ConversationRelay
                url="wss://matthias-nollek--twilio-voice-nemo-twiliochatbot-fastapi-app.modal.run/ws"
                welcomeGreeting="Hi! I'm Jane. Just chat with me!!">
            </ConversationRelay>
          </Connect>
        </Response>

In [36]:
!npm install -g wscat
!wscat -c wss://matthias-nollek--twilio-voice-nemo-twiliochatbot-fastapi-app.modal.run/ws
# Sende dann dieses JSON in die Konsole:
# {"type": "prompt", "voicePrompt": "Hallo, wie geht es dir?", "last": true}

⠙⠹⠸⠼
changed 9 packages in 702ms
Connected (press CTRL+C to quit)
> {"type": "prompt", "voicePrompt": "Hallo, wie geht es dir?", "last": true}
< {"type":"text","token":"","last":true}
> {"type": "prompt", "voicePrompt": "Hallo, wer bist du?", "last": true}
error: Invalid WebSocket frame: invalid status code 1006
> 